# Chapter 4 — Norms, Topology and Banach Spaces

Companion notebook to Chapter 4 of *Advanced Calculus for Machine Learning*.
It checks the numerical claims of the chapter with PyTorch. Everything runs
on a CPU in a few seconds.

Contents:

1. Dual norms and steepest descent directions
2. The parallelogram law, and the constants between norms
3. Norms on functions: not equivalent, not complete
4. Operator norms
5. The Lipschitz constant of a network; spectral normalization
6. The contraction mapping principle
7. Value iteration
8. Unbounded linear maps

In [1]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. Dual norms and steepest descent directions

Proposition 4.5: the dual norms of $\ell^1$, $\ell^2$, $\ell^\infty$ are
$\ell^\infty$, $\ell^2$, $\ell^1$. We check the formulas against a brute
force maximization of $\phi$ over random points of each unit sphere
(Exercise 4.1 uses $\phi = [2, -1, 3]$).

In [2]:
phi = torch.tensor([2.0, -1.0, 3.0])
samples = torch.randn(400000, 3)
spheres = {
    "l1  ": samples / samples.abs().sum(dim=1, keepdim=True),
    "l2  ": samples / samples.norm(dim=1, keepdim=True),
    "linf": samples / samples.abs().max(dim=1, keepdim=True).values,
}
formulas = {"l1  ": phi.abs().max(), "l2  ": phi.norm(), "linf": phi.abs().sum()}
maximizers = {"l1  ": torch.tensor([0.0, 0.0, 1.0]), "l2  ": phi / phi.norm(),
              "linf": torch.sign(phi)}
for name, pts in spheres.items():
    brute = (pts @ phi).max().item()
    exact = formulas[name].item()
    assert brute <= exact + 1e-12
    assert torch.allclose(maximizers[name] @ phi, formulas[name])
    print(f"{name}: dual norm {exact:.4f}  (best of 400000 random points: {brute:.4f}), "
          f"attained at {[round(v, 4) + 0.0 for v in maximizers[name].tolist()]}")

l1  : dual norm 3.0000  (best of 400000 random points: 2.9951), attained at [0.0, 0.0, 1.0]
l2  : dual norm 3.7417  (best of 400000 random points: 3.7416), attained at [0.5345, -0.2673, 0.8018]
linf: dual norm 6.0000  (best of 400000 random points: 5.9876), attained at [1.0, -1.0, 1.0]


Figure 4.1: for $\phi = [2, 1]$ on $\mathbb R^2$, the steepest ascent
directions are $(1, 0)$, $\tfrac1{\sqrt5}(2, 1)$ and $(1, 1)$, with values
$2$, $\sqrt5$ and $3$. Steepest descent for $\ell^\infty$ is sign descent.

In [3]:
gamma = torch.tensor([2.0, 1.0])
angles = torch.linspace(0, 2 * math.pi, 400001)
directions = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
norms = {"l1  ": lambda p: p.abs().sum(dim=1, keepdim=True),
         "l2  ": lambda p: p.norm(dim=1, keepdim=True),
         "linf": lambda p: p.abs().max(dim=1, keepdim=True).values}
for name, norm in norms.items():
    sphere = directions / norm(directions)
    values = sphere @ gamma
    best = sphere[values.argmax()]
    print(f"{name}: steepest ascent {[round(v, 3) + 0.0 for v in best.tolist()]}, "
          f"value {values.max().item():.4f}")
print("sign(gamma) =", torch.sign(gamma).tolist())

l1  : steepest ascent [1.0, 0.0], value 2.0000
l2  : steepest ascent [0.894, 0.447], value 2.2361
linf: steepest ascent [1.0, 1.0], value 3.0000
sign(gamma) = [1.0, 1.0]


## 2. The parallelogram law and the constants

Proposition 4.3: a norm that comes from an inner product satisfies
$\lVert u+v\rVert^2 + \lVert u-v\rVert^2 = 2\lVert u\rVert^2 +
2\lVert v\rVert^2$. With $u = e_1$, $v = e_2$ it fails for $\ell^1$ and
$\ell^\infty$.

In [4]:
e1, e2 = torch.tensor([1.0, 0.0]), torch.tensor([0.0, 1.0])
for name, p in [("l1  ", 1), ("l2  ", 2), ("linf", float("inf"))]:
    n = lambda x: torch.linalg.vector_norm(x, ord=p)
    lhs = n(e1 + e2) ** 2 + n(e1 - e2) ** 2
    rhs = 2 * n(e1) ** 2 + 2 * n(e2) ** 2
    print(f"{name}: {lhs.item():.1f} vs {rhs.item():.1f}")

l1  : 8.0 vs 4.0
l2  : 4.0 vs 4.0
linf: 2.0 vs 4.0


Proposition 4.19:
$\lVert x\rVert_\infty\le\lVert x\rVert_2\le\lVert x\rVert_1
\le\sqrt n\lVert x\rVert_2\le n\lVert x\rVert_\infty$, each inequality
sharp. The constants grow with the dimension.

In [5]:
n = 50
X = torch.randn(10000, n)
linf = X.abs().max(dim=1).values
l2, l1 = X.norm(dim=1), X.abs().sum(dim=1)
assert (linf <= l2).all() and (l2 <= l1).all()
assert (l1 <= math.sqrt(n) * l2 + 1e-12).all() and (math.sqrt(n) * l2 <= n * linf + 1e-12).all()
print("the chain of inequalities holds on 10000 random vectors of dimension", n)

for name, x in [("e_1        ", torch.eye(n)[0]), ("(1, ..., 1)", torch.ones(n))]:
    print(f"x = {name}: linf = {x.abs().max().item():.3f}, l2 = {x.norm().item():.3f}, "
          f"l1 = {x.abs().sum().item():.3f}")
print("for n = 10^8 parameters, sqrt(n) =", math.sqrt(1e8))

the chain of inequalities holds on 10000 random vectors of dimension 50
x = e_1        : linf = 1.000, l2 = 1.000, l1 = 1.000
x = (1, ..., 1): linf = 1.000, l2 = 7.071, l1 = 50.000
for n = 10^8 parameters, sqrt(n) = 10000.0


## 3. Norms on functions

Example 4.20: $f_k(t) = t^k$ on $[0, 1]$ has sup norm $1$ and
$\lVert f_k\rVert_1 = \tfrac1{k+1}$. Integrals are computed with the
trapezoidal rule on a fine grid.

In [6]:
t = torch.linspace(0, 1, 2000001)
for k in (1, 10, 100, 1000):
    f = t ** k
    print(f"k = {k:5d}: sup norm = {f.max().item():.3f}, "
          f"L1 norm = {torch.trapezoid(f, t).item():.6f}  (1/(k+1) = {1 / (k + 1):.6f})")

k =     1: sup norm = 1.000, L1 norm = 0.500000  (1/(k+1) = 0.500000)


k =    10: sup norm = 1.000, L1 norm = 0.090909  (1/(k+1) = 0.090909)
k =   100: sup norm = 1.000, L1 norm = 0.009901  (1/(k+1) = 0.009901)


k =  1000: sup norm = 1.000, L1 norm = 0.000999  (1/(k+1) = 0.000999)


Example 4.21 and Figure 4.2: the ramps $f_k$ form a Cauchy sequence for
$\lVert\cdot\rVert_1$ (here $\lVert f_l - f_k\rVert_1 =
\tfrac12(\tfrac1k - \tfrac1l)$), but stay at sup distance close to $1$
from each other when $l$ is much larger than $k$.

In [7]:
def ramp(k):
    return torch.clamp(k * (t - 0.5), 0.0, 1.0)


for k, l in [(2, 4), (10, 100), (100, 10000), (1000, 100000)]:
    diff = (ramp(l) - ramp(k)).abs()
    print(f"k = {k:5d}, l = {l:6d}: L1 distance = {torch.trapezoid(diff, t).item():.6f} "
          f"(formula {0.5 * (1 / k - 1 / l):.6f}),  sup distance = {diff.max().item():.3f}")

k =     2, l =      4: L1 distance = 0.125000 (formula 0.125000),  sup distance = 0.500


k =    10, l =    100: L1 distance = 0.045000 (formula 0.045000),  sup distance = 0.900
k =   100, l =  10000: L1 distance = 0.004950 (formula 0.004950),  sup distance = 0.990
k =  1000, l = 100000: L1 distance = 0.000495 (formula 0.000495),  sup distance = 0.990


## 4. Operator norms

Proposition 4.25 and Exercise 4.4: for $\ell^\infty$ on both sides the
operator norm is the largest row sum, for $\ell^1$ the largest column sum.
For $\ell^2$ it is the largest singular value. PyTorch calls these
`matrix_norm` with `ord=inf`, `ord=1` and `ord=2`.

In [8]:
A = torch.tensor([[1.0, -2.0], [3.0, 4.0]])
row_sum, col_sum = A.abs().sum(dim=1).max(), A.abs().sum(dim=0).max()
print("largest row sum:", row_sum.item(), "  matrix_norm(ord=inf):",
      torch.linalg.matrix_norm(A, ord=float("inf")).item())
print("largest column sum:", col_sum.item(), "  matrix_norm(ord=1):",
      torch.linalg.matrix_norm(A, ord=1).item())
assert row_sum == 7 and col_sum == 6

# brute force: sup of |Ax| / |x| over many directions
pts = torch.randn(400000, 2)
for name, p in [("linf", float("inf")), ("l1  ", 1), ("l2  ", 2)]:
    ratio = torch.linalg.vector_norm(pts @ A.T, ord=p, dim=1) / torch.linalg.vector_norm(pts, ord=p, dim=1)
    exact = torch.linalg.matrix_norm(A, ord=p).item()
    assert ratio.max().item() <= exact + 1e-12
    print(f"{name}: operator norm {exact:.4f}, largest ratio over random points {ratio.max().item():.4f}")

B = torch.randn(3, 2)
for p in (1, 2, float("inf")):
    assert torch.linalg.matrix_norm(B @ A, ord=p) <= torch.linalg.matrix_norm(B, ord=p) * torch.linalg.matrix_norm(A, ord=p) + 1e-12
print("|BA| <= |B| |A| for the three norms")

largest row sum: 7.0   matrix_norm(ord=inf): 7.0
largest column sum: 6.0   matrix_norm(ord=1): 6.0


linf: operator norm 7.0000, largest ratio over random points 7.0000


l1  : operator norm 6.0000, largest ratio over random points 6.0000
l2  : operator norm 5.1167, largest ratio over random points 5.1167
|BA| <= |B| |A| for the three norms


Proposition 4.24: between inner product spaces, the adjoint has the same
operator norm as the map. With Gram matrices $M = L_ML_M^\top$ and
$N = L_NL_N^\top$, the operator norm of $A$ is the spectral norm of
$L_N^\top AL_M^{-\top}$.

In [9]:
def random_spd(n):
    Bm = torch.randn(n, n)
    return Bm @ Bm.T + n * torch.eye(n)


def op_norm(A, M, N):
    LM, LN = torch.linalg.cholesky(M), torch.linalg.cholesky(N)
    return torch.linalg.matrix_norm(LN.T @ A @ torch.linalg.inv(LM).T, ord=2)


n, m = 4, 3
A, M, N = torch.randn(m, n), random_spd(n), random_spd(m)
A_star = torch.linalg.solve(M, A.T @ N)
print("|A| =", op_norm(A, M, N).item(), "  |A*| =", op_norm(A_star, N, M).item())
assert torch.allclose(op_norm(A, M, N), op_norm(A_star, N, M))

|A| = 2.703296070144211   |A*| = 2.7032960701442112


## 5. The Lipschitz constant of a network

Exercise 4.5: for $f(x) = W_2\,\mathrm{relu}(W_1x)$ with the matrices
below, the product of operator norms for $\ell^\infty$ is $4$, but
$f(x) = |x_1 - x_2|$ has Lipschitz constant $2$.

In [10]:
W1 = torch.tensor([[1.0, -1.0], [-1.0, 1.0]])
W2 = torch.tensor([[1.0, 1.0]])
f = lambda x: torch.relu(x @ W1.T) @ W2.T
bound = torch.linalg.matrix_norm(W2, ord=float("inf")) * torch.linalg.matrix_norm(W1, ord=float("inf"))
x, y = torch.randn(400000, 2), torch.randn(400000, 2)
assert torch.allclose(f(x).squeeze(1), (x[:, 0] - x[:, 1]).abs())
ratios = (f(x) - f(y)).abs().squeeze(1) / (x - y).abs().max(dim=1).values
print("product bound:", bound.item(), "  largest observed ratio:", round(ratios.max().item(), 4))

product bound: 4.0   largest observed ratio: 2.0


Spectral normalization divides each weight matrix by its spectral norm.
With PyTorch's parametrization the largest singular value of each layer
becomes (approximately, it is estimated by power iteration) $1$, and the
network is $1$-Lipschitz for $\ell^2$.

In [11]:
from torch.nn.utils.parametrizations import spectral_norm

net = torch.nn.Sequential(
    spectral_norm(torch.nn.Linear(8, 16), n_power_iterations=50),
    torch.nn.ReLU(),
    spectral_norm(torch.nn.Linear(16, 16), n_power_iterations=50),
    torch.nn.Tanh(),
    spectral_norm(torch.nn.Linear(16, 4), n_power_iterations=50),
)
net.train()
with torch.no_grad():
    for _ in range(5):                      # let the power iterations converge
        net(torch.randn(2, 8))
net.eval()
sigmas = [torch.linalg.matrix_norm(layer.weight, ord=2).item()
          for layer in net if isinstance(layer, torch.nn.Linear)]
print("spectral norms of the layers:", [round(s, 6) for s in sigmas])
with torch.no_grad():
    x, y = torch.randn(100000, 8), torch.randn(100000, 8)
    ratios = (net(x) - net(y)).norm(dim=1) / (x - y).norm(dim=1)
print("largest observed |f(x) - f(y)|_2 / |x - y|_2:", round(ratios.max().item(), 4))
assert ratios.max() <= 1.0 + 1e-6

spectral norms of the layers: [1.0, 1.0, 1.0]


largest observed |f(x) - f(y)|_2 / |x - y|_2: 0.2394


## 6. The contraction mapping principle

Figure 4.3: $f(x) = \tfrac12x + 1$ from $a_0 = 0$. Theorem 4.28 gives
$|a_k - a^*|\le\kappa^k|a_0 - a^*|$ and
$|a_k - a^*|\le\tfrac{\kappa^k}{1-\kappa}|a_1 - a_0|$.

In [12]:
f = lambda x: 0.5 * x + 1
kappa, a_star = 0.5, 2.0
a, a0, a1 = 0.0, 0.0, f(0.0)
for k in range(1, 9):
    a = f(a)
    err = abs(a - a_star)
    assert err <= kappa ** k * abs(a0 - a_star) + 1e-15
    assert err <= kappa ** k / (1 - kappa) * abs(a1 - a0) + 1e-15
    print(f"k = {k}: a_k = {a:.6f}, error = {err:.6f}, a-posteriori bound = "
          f"{kappa ** k / (1 - kappa) * abs(a1 - a0):.6f}")

k = 1: a_k = 1.000000, error = 1.000000, a-posteriori bound = 1.000000
k = 2: a_k = 1.500000, error = 0.500000, a-posteriori bound = 0.500000
k = 3: a_k = 1.750000, error = 0.250000, a-posteriori bound = 0.250000
k = 4: a_k = 1.875000, error = 0.125000, a-posteriori bound = 0.125000
k = 5: a_k = 1.937500, error = 0.062500, a-posteriori bound = 0.062500
k = 6: a_k = 1.968750, error = 0.031250, a-posteriori bound = 0.031250
k = 7: a_k = 1.984375, error = 0.015625, a-posteriori bound = 0.015625
k = 8: a_k = 1.992188, error = 0.007812, a-posteriori bound = 0.007812


The pitfall of Section 4.6 and Exercise 4.6: the map $x\mapsto Ax + b$ is
not a contraction for $\lVert\cdot\rVert_\infty$ but is one for
$\lVert x\rVert' = \max(|x_1|, 4|x_2|)$, with $\kappa = 0.75$. The fixed
point is $(6, 2)$ and $k = 58$ iterations guarantee an error below
$10^{-6}$.

In [13]:
A = torch.tensor([[0.5, 1.0], [0.0, 0.5]])
b = torch.tensor([1.0, 1.0])
D = torch.diag(torch.tensor([1.0, 4.0]))
print("operator norm for linf :", torch.linalg.matrix_norm(A, ord=float("inf")).item())
print("operator norm for |.|' :", torch.linalg.matrix_norm(D @ A @ torch.linalg.inv(D), ord=float("inf")).item())

x_star = torch.linalg.solve(torch.eye(2) - A, b)
print("fixed point:", x_star.tolist())
norm_prime = lambda x: (D @ x).abs().max().item()
x = torch.zeros(2)
first_step = norm_prime(A @ x + b - x)
for k in range(1, 59):
    x = A @ x + b
    assert norm_prime(x - x_star) <= 0.75 ** k / 0.25 * first_step + 1e-12
print("bound after 58 steps:", 16 * 0.75 ** 58, "  actual error:", norm_prime(x - x_star))
assert 16 * 0.75 ** 58 <= 1e-6 < 16 * 0.75 ** 57

operator norm for linf : 1.5
operator norm for |.|' : 0.75
fixed point: [6.0, 2.0]
bound after 58 steps: 9.071377899944184e-07   actual error: 8.881784197001252e-16


## 7. Value iteration

Exercise 4.7: two states, discount $\beta = \tfrac12$. The iterates of the
Bellman operator from $v = (0, 0)$ and the fixed point $v^* = (3, 6)$.

In [14]:
def bellman(v, r, P, beta):
    # r[s, u]: rewards; P[s, u, s']: transition probabilities
    return (r + beta * P @ v).max(dim=1).values


r = torch.tensor([[1.0, 0.0], [3.0, 0.0]])            # actions: stay, move
P = torch.tensor([[[1.0, 0.0], [0.0, 1.0]],
                  [[0.0, 1.0], [1.0, 0.0]]])
beta, v_star = 0.5, torch.tensor([3.0, 6.0])
assert torch.allclose(bellman(v_star, r, P, beta), v_star)
v = torch.zeros(2)
for k in range(1, 5):
    v = bellman(v, r, P, beta)
    print(f"k = {k}: T^k v = {v.tolist()}, error = {(v - v_star).abs().max().item()}")
assert torch.equal(v, torch.tensor([2.625, 5.625]))

k = 1: T^k v = [1.0, 3.0], error = 3.0
k = 2: T^k v = [1.5, 4.5], error = 1.5
k = 3: T^k v = [2.25, 5.25], error = 0.75
k = 4: T^k v = [2.625, 5.625], error = 0.375


Proposition 4.30 on a random problem with 6 states and 3 actions: the
Bellman operator shrinks sup-norm distances by at least the factor
$\beta$.

In [15]:
S, U, beta = 6, 3, 0.9
r = torch.randn(S, U)
P = torch.rand(S, U, S)
P = P / P.sum(dim=2, keepdim=True)
worst = 0.0
for _ in range(20000):
    v, w = 5 * torch.randn(S), 5 * torch.randn(S)
    ratio = (bellman(v, r, P, beta) - bellman(w, r, P, beta)).abs().max() / (v - w).abs().max()
    worst = max(worst, ratio.item())
print("largest |Tv - Tw|_inf / |v - w|_inf over 20000 pairs:", round(worst, 4), "<=", beta)
assert worst <= beta + 1e-12

largest |Tv - Tw|_inf / |v - w|_inf over 20000 pairs: 0.7985 <= 0.9


The choice of norm matters: with two states that both lead to state 1, the
linear part of $T$ is $\beta\begin{bmatrix}1&0\\1&0\end{bmatrix}$, of norm
$\beta$ for $\ell^\infty$ and $\beta\sqrt2$ for $\ell^2$.

In [16]:
beta = 0.9
Lin = beta * torch.tensor([[1.0, 0.0], [1.0, 0.0]])
print("linf operator norm:", torch.linalg.matrix_norm(Lin, ord=float("inf")).item())
print("l2 operator norm  :", torch.linalg.matrix_norm(Lin, ord=2).item(), "  beta*sqrt(2) =", beta * math.sqrt(2))

linf operator norm: 0.9
l2 operator norm  : 1.2727922061357857   beta*sqrt(2) = 1.2727922061357857


## 8. Unbounded linear maps

Example 4.27. Differentiation: $f_k(t) = \sin(2\pi kt)$ has sup norm $1$
and its derivative, computed here by autograd, has sup norm $2\pi k$.
Evaluation at $0$: the hat functions $h_k$ have $h_k(0) = 1$ and
$\lVert h_k\rVert_2^2 = \tfrac2{3k}$ (Exercise 4.9).

In [17]:
for k in (1, 10, 100):
    tt = torch.linspace(0, 1, 200001, requires_grad=True)
    fk = torch.sin(2 * math.pi * k * tt)
    (dfk,) = torch.autograd.grad(fk.sum(), tt)
    print(f"k = {k:3d}: sup|f_k| = {fk.abs().max().item():.3f}, "
          f"sup|f_k'| = {dfk.abs().max().item():.2f}  (2 pi k = {2 * math.pi * k:.2f})")

s = torch.linspace(-1, 1, 4000001)
for k in (1, 10, 100, 1000):
    h = torch.clamp(1 - k * s.abs(), min=0.0)
    l2_sq = torch.trapezoid(h ** 2, s).item()
    print(f"k = {k:4d}: h_k(0) = {h[len(s) // 2].item():.1f}, |h_k|_2^2 = {l2_sq:.6f} "
          f"(2/(3k) = {2 / (3 * k):.6f})")

k =   1: sup|f_k| = 1.000, sup|f_k'| = 6.28  (2 pi k = 6.28)
k =  10: sup|f_k| = 1.000, sup|f_k'| = 62.83  (2 pi k = 62.83)
k = 100: sup|f_k| = 1.000, sup|f_k'| = 628.32  (2 pi k = 628.32)
k =    1: h_k(0) = 1.0, |h_k|_2^2 = 0.666667 (2/(3k) = 0.666667)
k =   10: h_k(0) = 1.0, |h_k|_2^2 = 0.066667 (2/(3k) = 0.066667)
k =  100: h_k(0) = 1.0, |h_k|_2^2 = 0.006667 (2/(3k) = 0.006667)
k = 1000: h_k(0) = 1.0, |h_k|_2^2 = 0.000667 (2/(3k) = 0.000667)
